Extracción de la información relevante del Dataset de AI-KG.  Volcado a un csv con las columnas:
1. Relaciones
2. ID archivo origen
3. Hlight
4. Entidades marcadas en hlight

# 0. INICIALIZACION

In [1]:
import bz2
import shutil
import os
import xml.etree.ElementTree as ET
import csv
import html

In [ ]:
# folder = "DATA"
folder = "../Data"

files = [
    'res1',
    'res2',
    'res3',
    'res4',
    'res5-1',
    'res5-2',
    'res6-1',
    'res6-2',
    'res7-1',
    'res7-2',
    'res8-1',
    'res8-2',
    'res8-3',
    'res8-4'
    ]
files_comp = [
    'res1.xml.bz2',
    'res2.xml.bz2',
    'res3.xml.bz2',
    'res4.xml.bz2',
    'res5-1.xml.bz2',
    'res5-2.xml.bz2',
    'res6-1.xml.bz2',
    'res6-2.xml.bz2',
    'res7-1.xml.bz2',
    'res7-2.xml.bz2',
    'res8-1.xml.bz2',
    'res8-2.xml.bz2',
    'res8-3.xml.bz2',
    'res8-4.xml.bz2'
    ]

# 1. DESCOMPRESION

In [3]:
# for file in files_comp:
for file in files:
    print(f"-"*40)
    print(f"Comienza descompresion de archivo: {file}")
    
    file_comp = f"{file}.xml.bz2"
    file_in = os.path.join(folder, file_comp)

    file_desc = f"{file}.xml"
    file_out = os.path.join(folder, file_desc)
    
    
    with bz2.open(file_in, "rb") as f_in:
        with open(file_out, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
            
    print(f"Archivo {file} descomprimido")

----------------------------------------
Comienza descompresion de archivo: res1
Archivo res1 descomprimido
----------------------------------------
Comienza descompresion de archivo: res2
Archivo res2 descomprimido
----------------------------------------
Comienza descompresion de archivo: res3
Archivo res3 descomprimido
----------------------------------------
Comienza descompresion de archivo: res4
Archivo res4 descomprimido
----------------------------------------
Comienza descompresion de archivo: res5-1
Archivo res5-1 descomprimido
----------------------------------------
Comienza descompresion de archivo: res5-2
Archivo res5-2 descomprimido
----------------------------------------
Comienza descompresion de archivo: res6-1
Archivo res6-1 descomprimido
----------------------------------------
Comienza descompresion de archivo: res6-2
Archivo res6-2 descomprimido
----------------------------------------
Comienza descompresion de archivo: res7-1
Archivo res7-1 descomprimido
--------

# 2. LIMPIEZA XML

Dentro del apartado hlights no reconoce <b></b> como un nuevo nodo dentro de hlights.    
Estas son las entidades del abstract que son las que tengo que identificar.  

Al hacer print aparece:  
<hlight>has generated an &lt;b class="match term0"&gt;information&lt;/b&gt; &lt;b class="match term1"&gt;overload&lt;/b&gt;</hlight>

EN VEZ DE:  
<hlight>has generated an <b class="match term0">information</b> <b class="match term1">overload</b></hlight>

In [3]:
def limpiar_xml(input_xml, clean_xml):
    context = ET.iterparse(input_xml, events=("start", "end"))
    root = None

    for event, elem in context:
        if root is None:
            root = elem  # guardar raíz

        if event == "end" and elem.tag == "hlight":
            # desescapar solo el contenido de hlight
            texto = "".join(elem.itertext())
            texto_unescaped = html.unescape(texto)

            # eliminar contenido previo
            elem.clear()

            # volver a insertar como XML (desescapado)
            try:
                wrapper = ET.fromstring(f"<root>{texto_unescaped}</root>")
                for child in wrapper:
                    elem.append(child)
                if wrapper.text:
                    elem.text = wrapper.text
            except ET.ParseError:
                elem.text = texto_unescaped  # fallback

    tree = ET.ElementTree(root)
    tree.write(clean_xml, encoding="utf-8", xml_declaration=True)

In [4]:
for file in files:
    print(f"-"*40)
    print(f"Comienza limpieza de archivo: {file}")
    
    file_xml = f"{file}.xml"
    file_in = os.path.join(folder, file_xml)

    file_clean = f"{file}_clean.xml"
    file_out = os.path.join(folder, file_clean)
    
    
    limpiar_xml(file_in, file_out)
            
    print(f"Archivo {file} limpio")

----------------------------------------
Comienza limpieza de archivo: res1
Archivo res1 limpio
----------------------------------------
Comienza limpieza de archivo: res2
Archivo res2 limpio
----------------------------------------
Comienza limpieza de archivo: res3
Archivo res3 limpio
----------------------------------------
Comienza limpieza de archivo: res4
Archivo res4 limpio
----------------------------------------
Comienza limpieza de archivo: res5-1
Archivo res5-1 limpio
----------------------------------------
Comienza limpieza de archivo: res5-2
Archivo res5-2 limpio
----------------------------------------
Comienza limpieza de archivo: res6-1
Archivo res6-1 limpio
----------------------------------------
Comienza limpieza de archivo: res6-2
Archivo res6-2 limpio
----------------------------------------
Comienza limpieza de archivo: res7-1
Archivo res7-1 limpio
----------------------------------------
Comienza limpieza de archivo: res7-2
Archivo res7-2 limpio
----------------

# 3. VOLCADO DE xml A csv

De cada elemento <triple> en el xml se extrae:
- Relaciones
- ID archivo origen
- Hlight
- Entidades marcadas en hlight

In [5]:
def extraer_hlight_completo(text_elem):
    hlight = text_elem.find("hlight")
    if hlight is None:
        return ""

    # Extrae TODO el texto dentro de hlight (incluyendo <b>)
    return "".join(hlight.itertext()).strip()


In [6]:
def extraer_b_tags(text_elem):
    hlight = text_elem.find("hlight")
    if hlight is None:
        return ""

    resultados = []

    # for b in hlight.findall(".//b"):
    for b in hlight.findall("b"):
        if b.text:
            resultados.append(b.text.strip())

    return "|".join(resultados)


In [ ]:
def procesar_xml(input_xml, output_csv):
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["relaciones", "text_id", "hlight_full", "hlight_b"])

        context = ET.iterparse(input_xml, events=("start", "end"))

        relaciones_actuales = []

        for event, elem in context:

            # Nuevo triple - Nuevas relaciones
            if event == "start" and elem.tag == "triple":
                relaciones_actuales = []

            # Relaciones
            elif event == "end" and elem.tag == "rels":
                relaciones_actuales = [
                    (r.text or "").strip()
                    for r in elem.findall("r")
                ]
             
            elif event == "end" and elem.tag == "text":            
                text_id = elem.attrib.get("id", "")

                hlight_full = extraer_hlight_completo(elem)
                hlight_b = extraer_b_tags(elem)

                relaciones_str = "|".join(relaciones_actuales)

                writer.writerow([
                    relaciones_str,
                    text_id,
                    hlight_full,
                    hlight_b
                ])

                elem.clear()

            # Limpiar memoria
            elif event == "end" and elem.tag == "triple":
                elem.clear()


In [8]:
for file in files:
    print(f"-"*40)
    print(f"Comienza volcado de archivo {file} a csv")
    
    file_xml = f"{file}_clean.xml"
    file_in = os.path.join(folder, file_xml)

    file_csv = f"{file}.csv"
    file_out = os.path.join(folder, file_csv)
    
    
    procesar_xml(file_in, file_out)
            
    print(f"Archivo {file} volcado")

----------------------------------------
Comienza volcado de archivo res1 a csv
Archivo res1 volcado
----------------------------------------
Comienza volcado de archivo res2 a csv
Archivo res2 volcado
----------------------------------------
Comienza volcado de archivo res3 a csv
Archivo res3 volcado
----------------------------------------
Comienza volcado de archivo res4 a csv
Archivo res4 volcado
----------------------------------------
Comienza volcado de archivo res5-1 a csv
Archivo res5-1 volcado
----------------------------------------
Comienza volcado de archivo res5-2 a csv
Archivo res5-2 volcado
----------------------------------------
Comienza volcado de archivo res6-1 a csv
Archivo res6-1 volcado
----------------------------------------
Comienza volcado de archivo res6-2 a csv
Archivo res6-2 volcado
----------------------------------------
Comienza volcado de archivo res7-1 a csv
Archivo res7-1 volcado
----------------------------------------
Comienza volcado de archivo re

# 4. Visualizacion del csv

In [ ]:
def comprobar_csv(csv_file, n=10):
    with open(csv_file, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            print(row)
            if i >= n:
                break

In [ ]:
csv = "res1_sample.csv"
file_csv = os.path.join(folder, file_csv)

comprobar_csv(file_csv, 10)
